# Chapter 17 — TabICL Family

Reproduces:
- Figure 17.1: Per-head energy split (alpha_S, alpha_A) for TabICL-lite
  trained on the same SCM prior as TabPFN-lite (Chapter 16).
- Figure 17.2: Side-by-side per-layer alpha_A comparison: TabICL-lite vs
  TabPFN-lite. Same prior, same hyperparameters, same training schedule —
  only the attention non-linearity differs.
- Table 17.2: Side-by-side numeric summary.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from tabkernels.architectures import TabICLLite, TabPFNLite
from tabkernels.priors import SCMPrior, SCMConfig
from tabkernels.training import PFNTrainer
from tabkernels.transparency import decompose_attention, kernel_energy_split

torch.manual_seed(0); np.random.seed(0)
_p = os.getcwd()
while _p and not os.path.isdir(os.path.join(_p, 'affinity', 'book')):
    _p = os.path.dirname(_p)
FIGURES_DIR = os.path.join(_p, 'affinity', 'book', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)


## Train TabICL-lite on the same prior as TabPFN-lite

Identical hyperparameters to Chapter 16's TabPFN-lite run: $d_\text{model}=64$,
$4$ heads, $2$ layers, $1500$ steps, Adam $3 \times 10^{-3}$. The only
architectural difference is the attention non-linearity: TabICL-lite uses
linear attention with $\phi(z) = \text{elu}(z)+1$ followed by row
normalisation; TabPFN-lite uses softmax. The bilinear form $B = W_Q^\top W_K$
is parameterised identically in both, so the post-hoc decomposition
diagnostic compares apples to apples.


In [ ]:
D = 4
prior = SCMPrior(SCMConfig(structural='mlp', edge_prob=0.5, noise_scale=0.3,
                           mlp_hidden=12))

torch.manual_seed(0)
tabicl = TabICLLite(d_in=D, d_model=64, n_heads=4, n_layers=2, dim_ff=128)
trainer_icl = PFNTrainer(prior=prior, model=tabicl, n_steps=1500,
                         n_ctx=48, n_query=24, d=D, lr=3e-3,
                         eval_every=100, seed=1)
state_icl = trainer_icl.train()
print(f'TabICL-lite final train loss : {state_icl.losses[-1]:.4f}')
print(f'TabICL-lite final eval  loss : {state_icl.eval_losses[-1]:.4f}')

# Re-train TabPFN-lite with the same seed for an apples-to-apples comparison.
torch.manual_seed(0)
tabpfn = TabPFNLite(d_in=D, d_model=64, n_heads=4, n_layers=2, dim_ff=128)
trainer_pfn = PFNTrainer(prior=prior, model=tabpfn, n_steps=1500,
                         n_ctx=48, n_query=24, d=D, lr=3e-3,
                         eval_every=100, seed=1)
state_pfn = trainer_pfn.train()
print(f'TabPFN-lite final train loss : {state_pfn.losses[-1]:.4f}')
print(f'TabPFN-lite final eval  loss : {state_pfn.eval_losses[-1]:.4f}')


## Held-out ICL inference comparison

In [ ]:
def held_out_mse(model, n=20):
    model.eval()
    mses = []
    with torch.no_grad():
        for k in range(n):
            X_c, y_c, X_q, y_q = prior.sample_episode(48, 24, D, seed=20000 + k)
            yhat = model(X_q, X_c, y_c)
            mses.append((((yhat - y_q) ** 2).mean() / y_q.var().clamp_min(1e-3)).item())
    return float(np.mean(mses)), float(np.std(mses) / np.sqrt(len(mses)))


m_icl, s_icl = held_out_mse(tabicl)
m_pfn, s_pfn = held_out_mse(tabpfn)
print(f'TabICL-lite held-out norm-MSE : {m_icl:.3f} +/- {s_icl:.3f}')
print(f'TabPFN-lite held-out norm-MSE : {m_pfn:.3f} +/- {s_pfn:.3f}')


## Figure 17.1 — Per-head energy split for TabICL-lite

Stacked-bar plot of $\alpha_S$ and $\alpha_A$ for every head of every
layer of TabICL-lite. Compare to Figure 16.1 (TabPFN-lite).


In [ ]:
def per_head_energy(model):
    rows = []
    for li, block in enumerate(model.attention_blocks()):
        d = decompose_attention(block)
        for h in range(d['B'].shape[0]):
            a_s, a_a = kernel_energy_split(d['B'][h])
            rows.append((li, h, a_s, a_a))
    return rows


rows_icl = per_head_energy(tabicl)
rows_pfn = per_head_energy(tabpfn)

print('TabICL-lite per-head:')
print('Layer  Head  alpha_S   alpha_A')
for li, h, a_s, a_a in rows_icl:
    print(f'{li:>5d}  {h:>4d}  {a_s:>7.3f}  {a_a:>7.3f}')

mean_s_icl = float(np.mean([r[2] for r in rows_icl]))
mean_a_icl = float(np.mean([r[3] for r in rows_icl]))
mean_s_pfn = float(np.mean([r[2] for r in rows_pfn]))
mean_a_pfn = float(np.mean([r[3] for r in rows_pfn]))
print(f'\nTabICL-lite mean alpha_S/alpha_A : {mean_s_icl:.3f} / {mean_a_icl:.3f}')
print(f'TabPFN-lite mean alpha_S/alpha_A : {mean_s_pfn:.3f} / {mean_a_pfn:.3f}')

fig, ax = plt.subplots(1, 1, figsize=(7, 3.8))
xs = np.arange(len(rows_icl))
labels = [f'L{li}H{h}' for li, h, _, _ in rows_icl]
sym = [r[2] for r in rows_icl]; asym = [r[3] for r in rows_icl]
ax.bar(xs, sym, color='C0', label=r'$\alpha_S$')
ax.bar(xs, asym, bottom=sym, color='C3', label=r'$\alpha_A$')
ax.axhline(0.5, color='gray', ls='--', alpha=0.5)
ax.set_xticks(xs); ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_ylabel('energy fraction'); ax.set_ylim(0, 1.05)
ax.set_title('Figure 17.1: TabICL-lite per-head energy split')
ax.legend(loc='upper right')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_17_01_tabicl_energy_split.pdf', bbox_inches='tight')
plt.show()


## Figure 17.2 — Side-by-side TabICL vs TabPFN

Per-head $\alpha_A$ for both architectures, plotted as paired bars. The
question this figure answers: do the two FMs allocate skew-symmetric energy
similarly under identical training conditions?


In [ ]:
xs = np.arange(len(rows_icl))
asym_icl = [r[3] for r in rows_icl]
asym_pfn = [r[3] for r in rows_pfn]
labels = [f'L{li}H{h}' for li, h, _, _ in rows_icl]
width = 0.4

fig, ax = plt.subplots(1, 1, figsize=(7, 3.8))
ax.bar(xs - width/2, asym_pfn, width, color='C2', label='TabPFN-lite (softmax)')
ax.bar(xs + width/2, asym_icl, width, color='C1', label='TabICL-lite (linear)')
ax.axhline(np.mean(asym_pfn), color='C2', ls='--', alpha=0.5)
ax.axhline(np.mean(asym_icl), color='C1', ls='--', alpha=0.5)
ax.set_xticks(xs); ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_ylabel(r'$\alpha_A$ (skew-symmetric energy)')
ax.set_title('Figure 17.2: Per-head $\\alpha_A$, TabPFN-lite vs TabICL-lite')
ax.legend(loc='upper right'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_17_02_tabicl_vs_tabpfn.pdf', bbox_inches='tight')
plt.show()
print(f'mean alpha_A: TabPFN-lite {np.mean(asym_pfn):.3f}, TabICL-lite {np.mean(asym_icl):.3f}')
